# Morris sensitivity Analysis (SALib)

In [1]:
import os
import sys
import logging
from tqdm import tqdm
import shutil
import json
from pathlib import Path


import ogstools as ot
import pyvista as pv
import pandas as pd
import numpy as np
from scipy import integrate
from SALib.sample import morris as morris_sampler
from SALib.analyze import morris as morris_analyzer

from meshing import create_rectangle_frac_mesh_v3
from functions_s import save_combined_mesh

In [ ]:
#functions---------------------------------------

def run_simulation_OGS(prj_in, prj_out,factors,MESH_DIR,RUN_DIR,coords):
    try:
        temp_prj(prj_in, prj_out, factors)
        model = ot.Project(input_file=prj_out,verbose=False)
        model.run_model(background=False,args=f"-m {MESH_DIR} -o {RUN_DIR}", logfile="SA_log.txt",write_logs=False)
        return extract_values(RUN_DIR, coords)
    except Exception as e:
        print(f"Simulation failed with params {factors}: {e}")
        return None 

def sens_index(result):

    if result is None or 'values' not in result or len(result['values']) == 0:
            return 0.0, 0.0, 0.0
        
    p = np.array(result['values'])
    t = np.array(result['timevalues'])
    
    sum_c = integrate.simpson(y=p, x=t) if len(p) > 1 else 0.0
    peak = np.max(p)
    p_end = p[-1]

    return sum_c, peak, p_end


def extract_values(RUN_DIR,coords): 

    pvd_files = list(Path(RUN_DIR).glob("*.pvd"))
    if not pvd_files:
        raise FileNotFoundError(f"No .pvd file found in {RUN_DIR}")
    
    pvd_path = pvd_files[0]
    ms = ot.MeshSeries(pvd_path)
    pressure=ot.variables.pressure
    pressure= pressure.replace(output_unit="MPa")
    
    ms_probes=ms.probe(points=coords)
    raw_pressure_array = ms_probes['pressure']
    clean_p = np.squeeze(raw_pressure_array) #or raw_pressure_array()[:,0] taking the one point scalar

    data_bundle = {
        'values': clean_p, 
        'timevalues': np.array(ms.timevalues),
        'metadata': {
            'variable_name': 'pressure',
            'unit': 'MPa',
            'time_unit': 's',
            'coordinates': coords,
            'source_file': str(pvd_path)
        }
    }

    return data_bundle



def Morris_sample(names,bounds,N=50,num_levels=4):

    problem = {'num_vars': len(names), 
            'names': names,
            'bounds': bounds}

    param_values = morris_sampler.sample(problem, N=N, num_levels=num_levels)

    return param_values


def calculate_keff(factors):
    pjack=factors['pjack']
    p1=factors['p1']
    p2=factors['p2']
    wr=max(factors['wr'], 1e-6)
    k01=factors['k01']
    k02=factors['k02']

    prev=np.linspace(p1,p2,50)

    tanh_term = np.tanh((prev - pjack) / wr)
    c=(k02 - k01) * 0.5
    k_value = k01 + c* (1 + tanh_term)

    sf0=factors['sf0']
    beta_dimen=factors['b_dim']
    beta=pjack*beta_dimen

    X= np.sqrt(3)*np.sqrt(k_value)/k_value
    Y= c*(1-tanh_term**2)/wr
    s_value=  sf0 + beta*X*Y

    keff=k_value*s_value/sf0
    return keff


# def temp_prj(prj_in, prj_out,factors): 

#     keff=factors['keff']
#     kmatrix=factors['k01'] #considering kma=k01, kfrac and kma start from same basis
#     smatrix=factors['sma']

#     values_str = " ".join(map(str, keff))

#     model = ot.Project(input_file=prj_in,output_file=prj_out)
#     xpath='./curves/curve[name="k_curve"]/values'
#     medium=0

#     try:
#         model.replace_text(values_str,xpath)
#         model.replace_medium_property_value(medium,'permeability',kmatrix) 
#         model.replace_medium_property_value(medium,'storage',smatrix)

#     except Exception as e:
#         print(f"CRITICAL ERROR in PRJ update: {e}")
#         raise # Stop the loop if the input file is not correctly updated

#     model.write_input()

def temp_prj(prj_in, prj_out,factors): 

    keff=factors['keff']
    sfrac=factors['sf0'] 

    values_str = " ".join(map(str, keff))

    model = ot.Project(input_file=prj_in,output_file=prj_out)
    xpath='./curves/curve[name="k_curve"]/values'
    medium=1

    try:
        model.replace_text(values_str,xpath)
        model.replace_medium_property_value(medium,'storage',sfrac)

    except Exception as e:
        print(f"CRITICAL ERROR in PRJ update: {e}")
        raise # Stop the loop if the input file is not correctly updated

    model.write_input()

def mesh_gen(MSH_FILE,MESH_DIR,geom_data,factors):
    L=factors['L']
    thickness=geom_data["h"]
    r_well=geom_data["r_well"]
    mesh_size=thickness/4.0
    refine_well=thickness/20.0 #thickness/20
    refine_frac=thickness/30.0 #thickness/30 #smaller than well refinining

    create_rectangle_frac_mesh_v3(
        MSH_FILE,
        radius= 100.0,
        height= thickness,
        mesh_size= mesh_size,
        center_z=-40.6,
        r_well = r_well,
        length = L,
        refine_well = refine_well,  # Element size at the well
        refine_frac = refine_frac   # Element size along the fracture
    ) 

    meshes = ot.Meshes.from_gmsh(MSH_FILE, log=False)
    for name, mesh in meshes.items():
        vtu_path = (MESH_DIR / f"rectangle_{name}.vtu").as_posix()
        pv.save_meshio(vtu_path, mesh)
        # print(f"Saved {vtu_path}") #hiding in multiple mesh generating evaluaiton, also from function_S

    combined_vtu = (MESH_DIR / "combined_fracture_mesh.vtu").as_posix()
    save_combined_mesh(MSH_FILE, combined_vtu)

In [ ]:
#pre-conditions-------------------------------------------
cwd=Path.cwd()

OGS_PATH = r"C:\OGS_Exc\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin"

if OGS_PATH is not None:
    os.environ["OGS_BIN_PATH"] = OGS_PATH
OUT_DIR = Path(os.environ.get("OGS_TESTRUNNER_OUT_DIR", "_out_t"))
MESH_DIR = OUT_DIR / "mesh"
RUN_DIR=OUT_DIR / "run"
RESULTS_DIR = OUT_DIR / "results"
npy_dir = RESULTS_DIR / "raw_data"


shutil.rmtree(OUT_DIR, ignore_errors=True)
MESH_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
npy_dir.mkdir(parents=True, exist_ok=True)

log_path=cwd/'simulation_run.log'
logging.basicConfig(
    filename=str(log_path),
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

tqdm.monitor_interval = 0 #hide OGS probes loading bar


prj_in=cwd/'BH10_20180718_40.6_SA_v2.prj'
prj_out=cwd/'_out_t/BH10_20180718_40.6_SA_temp.prj'

factors={
         'sf0':8.5e-4,
         'pjack':1.0,
         'wr':1.0,
         'L':1.0,
         'k01':1.0,
         'k02':1.0,
         'sma':1.0,
         'b_dim':1.0,
         'p1':1.0,
         'p2':5.0e6,
         'keff':1.0
}

y=-40.6
r_st=0.038 
coords = np.array([[r_st, y, 1e-18]])
labels= f"OGS: r={r_st:>5.3f}, y={y}" 

names=list(factors.keys())
names=names[:-7]
bounds=[[2.0926206997084548e-10,8.5e-4],
                        [3.1e6,3.6e6],
                        [0.2e6,0.5e6],
                        [0.5,20]
                        ]

MSH_FILE = MESH_DIR / "symmetric_cylinder_3D.msh"
geom_data={"h":0.7, #mesh as in field data
            "r_well":0.038
}


In [ ]:
#generating the sampling and loading/updating register and factors

version = "v12" 
filename = f"morris_samples_{version}.csv"
archive_file = RESULTS_DIR / f"results_archive_{version}.jsonl"

if not os.path.exists(filename):
    X = Morris_sample(names, bounds, N=30, num_levels=4) #section to change the number of directions N and levels
    pd.DataFrame(X, columns=names).to_csv(filename, index=False)

    with open(archive_file, "w") as f:
        pass 
    print(f"Created new archive: {archive_file}")

if not os.path.exists(archive_file):
    with open(archive_file, "w") as f:
        pass # Creates an empty file
    print(f"Initialized empty archive: {archive_file}")

df_X = pd.read_csv(filename)
archive_file = archive_file

processed_indices = set()
with open(archive_file, "r") as f:
    for line in f:
        try:
            entry = json.loads(line)
            processed_indices.add(entry["index"])
        except json.JSONDecodeError:
            continue 

static_factors = {
         'k01':2e-15,
         'k02':5e-10,
         'sma':2.0926206997084548e-10,
         'b_dim':3.0,
         'p1':1.0,
         'p2':5.0e6,
         'keff':1.0
}

df_full = df_X.copy()
for key, value in static_factors.items():
    df_full[key] = value

Created new archive: _out_t\results\results_archive_v10.jsonl


In [5]:
#exectution of OGS-morris: sensitivity indexes and binary results


for i in tqdm(range(len(df_full)), desc="Overall Progress", unit="run"):
    if i in processed_indices:
        continue
    
    factors = df_full.iloc[i].to_dict()
    factors['keff'] = calculate_keff(factors)
    
    if RUN_DIR.exists(): shutil.rmtree(RUN_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    
    mesh_gen(MSH_FILE,MESH_DIR,geom_data,factors)
    
    try:
        result = run_simulation_OGS(prj_in, prj_out, factors, MESH_DIR, RUN_DIR, coords)
        
        if result is not None:
            np.save(npy_dir / f"raw_run_{i}.npy", result) #binary saving per run
            sum_c, peak, p_end = sens_index(result)
            
            logging.info(f"Run {i} successful | Peak={peak:.2e}")
        else:
            logging.warning(f"Run {i} returned None (Simulation Failure)")
            sum_c, peak, p_end = None, None, None
          
    except Exception as e:
        logging.error(f"Run {i} crashed: {str(e)}")
        sum_c, peak, p_end = None, None, None

    with open(archive_file, "a") as f:
        f.write(json.dumps({"index": i, "sum_c": sum_c, "peak": peak, "p_end": p_end}) + "\n")

Overall Progress:   0%|          | 0/10 [00:00<?, ?run/s]


CMD: D:\OGS_Executable\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

Combined mesh saved to: _out_t/mesh/combined_fracture_mesh.vtu


Overall Progress:  10%|█         | 1/10 [01:35<14:15, 95.00s/run]


CMD: D:\OGS_Executable\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

Combined mesh saved to: _out_t/mesh/combined_fracture_mesh.vtu


Overall Progress:  20%|██        | 2/10 [05:39<24:24, 183.12s/run]

...

Overall Progress:  80%|████████  | 8/10 [14:06<03:08, 94.24s/run]


CMD: D:\OGS_Executable\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

Combined mesh saved to: _out_t/mesh/combined_fracture_mesh.vtu


Overall Progress:  90%|█████████ | 9/10 [35:23<07:43, 463.78s/run]


CMD: D:\OGS_Executable\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

Combined mesh saved to: _out_t/mesh/combined_fracture_mesh.vtu


Overall Progress: 100%|██████████| 10/10 [1:07:56<00:00, 407.64s/run]
